# 1c. Model vs Real Data Comparison (BE & SC)

**Purpose**: Do the BE and SC models' predicted learning trajectories match
real animals? Putting synthetic and real data on the same axes for both models.

**Sections**:
1. Generate synthetic trajectories (BE and SC, multiple animals)
2. Load real data
3. Stat trajectory overlay (BE vs SC vs real)
4. Update matrix comparison at matched phases
5. Expert-phase stat distributions


## 1. Configuration & Imports

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import mannwhitneyu
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from Models.BE_core import BEParams, BEState, BEModel
from Models.SC_core import SCParams, SCState, SCModel
from behav_utils.data.loading import load_experiment
from behav_utils.data.synthetic import generate_synthetic_animal, sample_stimuli
from behav_utils.analysis.summary_stats import (
    compute_summary_stats,
    list_available_stats, FEATURE_MATRIX_STATS,
)
from behav_utils.analysis.session_features import build_feature_matrix_multi
from behav_utils.analysis.update_matrix import compute_update_matrix
from behav_utils.plotting.psychometric import plot_psychometric
from behav_utils.plotting.update_matrix import plot_update_matrix

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
DATA_PATH = '/Users/Serkan/Desktop/pro/PhD/main/repos/sound_categorisation/config.yaml'
STAGE = 'Full_Task_Cont'
MIN_SESSIONS = 15

N_SYNTHETIC = 5
N_SESSIONS_SYN = 30
TRIALS_PER_SESSION = 350
BURN_IN = 100

COMPARE_STATS = [
    'accuracy', 'psychometric_gof',
    'pse', 'slope', 'lapse_low', 'lapse_high',
    'recency', 'win_stay_rate', 'stimulus_sensitivity',
    'choice_entropy', 'perseveration', 'side_bias',
    'hard_accuracy', 'easy_accuracy',
    'w_stimulus', 'w_prev_choice_1',
]

## 2. Generate Synthetic Learning Trajectories

In [ ]:
# ── BE simulator: wraps the actual BE model ─────────────────────────────────
def be_simulator(stimuli, categories, rng, 
                 sigma_percep=0.15, A_repulsion=0.1,
                 eta_learning=0.35, eta_relax=0.12, **kwargs):
    """Simulate one session using the BE model."""
    params = BEParams(
        sigma_percep=sigma_percep, A_repulsion=A_repulsion,
        eta_learning=eta_learning, eta_relax=eta_relax,
    )
    state = BEModel.create_initial_state(
        params=params, burn_in=BURN_IN,
        seed=rng.integers(0, 2**31),
    )
    choices, _, _, _ = BEModel.simulate_session(
        params, state, stimuli, categories, rng,
    )
    return choices.astype(float)


def sc_simulator(stimuli, categories, rng,
                 sigma_percep=0.15, A_repulsion=0.1,
                 gamma=0.95, sigma_update=0.3, **kwargs):
    """Simulate one session using the SC model."""
    params = SCParams(
        sigma_percep=sigma_percep, A_repulsion=A_repulsion,
        gamma=gamma, sigma_update=sigma_update,
    )
    state = SCModel.create_initial_state(
        params=params, burn_in=BURN_IN,
        seed=rng.integers(0, 2**31),
    )
    choices, _, _, _ = SCModel.simulate_session(
        params, state, stimuli, categories, rng,
    )
    return choices.astype(float)


# ── Parameter trajectories: naive → expert ──────────────────────────────────
def be_param_trajectory(n_sessions, eta_start=0.35, eta_end=0.05):
    """Build per-session kwargs for BE: eta_learning decays exponentially."""
    rate = -np.log(eta_end / eta_start) / max(n_sessions - 1, 1)
    return [
        {
            'sigma_percep': 0.15,
            'A_repulsion': 0.1,
            'eta_learning': eta_start * np.exp(-rate * s),
            'eta_relax': 0.12,
        }
        for s in range(n_sessions)
    ]


def sc_param_trajectory(n_sessions, gamma_start=0.80, gamma_end=0.98):
    """Build per-session kwargs for SC: gamma increases (less updating)."""
    gammas = np.linspace(gamma_start, gamma_end, n_sessions)
    return [
        {
            'sigma_percep': 0.15,
            'A_repulsion': 0.1,
            'gamma': gammas[s],
            'sigma_update': 0.3,
        }
        for s in range(n_sessions)
    ]


# ── Generate BE synthetic animals ───────────────────────────────────────────
be_animals = []
for i in range(N_SYNTHETIC):
    animal, info = generate_synthetic_animal(
        animal_id=f'BE_SYN{i:02d}',
        n_sessions=N_SESSIONS_SYN,
        trials_per_session=TRIALS_PER_SESSION,
        seed=42 + i,
        simulator=be_simulator,
        per_session_simulator_kwargs=be_param_trajectory(N_SESSIONS_SYN),
        stage=STAGE,
    )
    be_animals.append(animal)
print(f"Generated {len(be_animals)} BE synthetic animals")

# ── Generate SC synthetic animals ───────────────────────────────────────────
sc_animals = []
for i in range(N_SYNTHETIC):
    animal, info = generate_synthetic_animal(
        animal_id=f'SC_SYN{i:02d}',
        n_sessions=N_SESSIONS_SYN,
        trials_per_session=TRIALS_PER_SESSION,
        seed=42 + i,
        simulator=sc_simulator,
        per_session_simulator_kwargs=sc_param_trajectory(N_SESSIONS_SYN),
        stage=STAGE,
    )
    sc_animals.append(animal)
print(f"Generated {len(sc_animals)} SC synthetic animals")

In [ ]:
# ── Build feature matrices ─────────────────────────────────────────────────
ALL_STAT_NAMES = list_available_stats()

df_be = build_feature_matrix_multi(be_animals, stage=None,
                                    stat_names=ALL_STAT_NAMES,
                                    compute_deltas=False)
df_be['source'] = 'BE'
print(f"BE feature matrix: {len(df_be)} sessions × {len(df_be.columns)} columns")

df_sc = build_feature_matrix_multi(sc_animals, stage=None,
                                    stat_names=ALL_STAT_NAMES,
                                    compute_deltas=False)
df_sc['source'] = 'SC'
print(f"SC feature matrix: {len(df_sc)} sessions × {len(df_sc.columns)} columns")

## 3. Load Real Data

In [ ]:
try:
    experiment = load_experiment(Path(DATA_PATH))
    real_animals = experiment.get_animals(min_sessions=MIN_SESSIONS, stage=STAGE)
    df_real = build_feature_matrix_multi(real_animals, stage=STAGE,
                                          stat_names=ALL_STAT_NAMES,
                                          compute_deltas=False)
    df_real['source'] = 'Real'
    df_real_trunc = df_real[df_real['session_idx'] < N_SESSIONS_SYN].copy()
    print(f"Real data: {len(real_animals)} animals, {len(df_real_trunc)} sessions (truncated)")
    HAS_REAL = True
except Exception as e:
    print(f"Could not load real data: {e}")
    print("Proceeding with synthetic-only comparison.")
    HAS_REAL = False
    df_real_trunc = pd.DataFrame()

## 4. Stat Trajectory Overlay

Blue = BE synthetic, Orange = SC synthetic, Grey/Black = real (if available).

In [ ]:
available = [s for s in COMPARE_STATS if s in df_be.columns and s in df_sc.columns]
if HAS_REAL:
    available = [s for s in available if s in df_real_trunc.columns]

n_stats = len(available)
n_cols = 4
n_rows = int(np.ceil(n_stats / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
axes_flat = np.array(axes).flatten()
fig.suptitle('Learning Trajectories: BE (blue) vs SC (orange) vs Real (grey)',
             fontsize=14, fontweight='bold', y=1.01)

for ax, sname in zip(axes_flat, available):
    # Real individual animals (grey)
    if HAS_REAL:
        for aid in df_real_trunc['animal_id'].unique():
            sub = df_real_trunc[df_real_trunc['animal_id'] == aid]
            ax.plot(sub['session_idx'], sub[sname], color='grey', alpha=0.3, lw=0.8)
        real_mean = df_real_trunc.groupby('session_idx')[sname].mean()
        ax.plot(real_mean.index, real_mean.values, 'k-', lw=2, label='Real mean')

    # BE individual animals
    for aid in df_be['animal_id'].unique():
        sub = df_be[df_be['animal_id'] == aid]
        ax.plot(sub['session_idx'], sub[sname], color='steelblue', alpha=0.3, lw=0.8)
    be_mean = df_be.groupby('session_idx')[sname].mean()
    ax.plot(be_mean.index, be_mean.values, color='steelblue', lw=2, label='BE mean')

    # SC individual animals
    for aid in df_sc['animal_id'].unique():
        sub = df_sc[df_sc['animal_id'] == aid]
        ax.plot(sub['session_idx'], sub[sname], color='darkorange', alpha=0.3, lw=0.8)
    sc_mean = df_sc.groupby('session_idx')[sname].mean()
    ax.plot(sc_mean.index, sc_mean.values, color='darkorange', lw=2, label='SC mean')

    ax.set_title(sname, fontsize=10)
    ax.set_xlabel('Session', fontsize=8)
    ax.tick_params(labelsize=7)

axes_flat[0].legend(fontsize=7)
for j in range(len(available), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.show()

## 5. Update Matrix Comparison at Matched Phases

In [ ]:
PHASES = {
    'Naive':  (0, 4),
    'Mid':    (10, 15),
    'Expert': (-5, None),
}

def get_phase_sessions(animal, stage, phase_range):
    sessions = animal.get_sessions(stage=stage) if stage else animal.sessions
    start, end = phase_range
    if start < 0:
        start = max(0, len(sessions) + start)
    if end is None:
        end = len(sessions)
    return sessions[start:end]


def mean_update_matrix(sessions):
    ums = []
    for sess in sessions:
        arrays = sess.trials.get_arrays(exclude_abort=True)
        um = compute_update_matrix(
            arrays['stimuli'], arrays['choices'], arrays['categories'], n_bins=8)
        um = um[0] if isinstance(um, tuple) else um  # Handle tuple output
        if um is not None and not np.all(np.isnan(um)):
            ums.append(um)
    return np.nanmean(ums, axis=0) if ums else None

# Compute for all sources
all_ums = {}
sources = [('BE', be_animals, None), ('SC', sc_animals, None)]
if HAS_REAL:
    sources.append(('Real', real_animals, STAGE))

for source_name, animals, stage in sources:
    for phase_name, phase_range in PHASES.items():
        phase_sessions = []
        for a in animals:
            phase_sessions.extend(get_phase_sessions(a, stage, phase_range))
        um = mean_update_matrix(phase_sessions) if phase_sessions else None
        all_ums[(source_name, phase_name)] = um

# Plot
n_sources = len(sources)
fig, axes = plt.subplots(n_sources, len(PHASES),
                          figsize=(4 * len(PHASES), 3.5 * n_sources))
if n_sources == 1:
    axes = axes.reshape(1, -1)
fig.suptitle('Update Matrices by Phase and Model', fontsize=13, fontweight='bold')

for row, (source_name, _, _) in enumerate(sources):
    for col, phase_name in enumerate(PHASES):
        ax = axes[row, col]
        um = all_ums.get((source_name, phase_name))
        if um is not None:
            plot_update_matrix(um, ax=ax)
        ax.set_title(f'{source_name} — {phase_name}', fontsize=10)

plt.tight_layout()
plt.show()

## 6. Expert-Phase Stat Distributions

In [ ]:
expert_be = df_be[df_be['session_idx'] >= N_SESSIONS_SYN - 5]
expert_sc = df_sc[df_sc['session_idx'] >= N_SESSIONS_SYN - 5]

available = [s for s in COMPARE_STATS if s in expert_be.columns and s in expert_sc.columns]
if HAS_REAL:
    expert_real = df_real_trunc[df_real_trunc['session_idx'] >= N_SESSIONS_SYN - 5]
    available = [s for s in available if s in expert_real.columns]

n_stats = len(available)
n_cols = 4
n_rows = int(np.ceil(n_stats / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes_flat = np.array(axes).flatten()
fig.suptitle('Expert Phase: Stat Distributions', fontsize=13, fontweight='bold')

for ax, sname in zip(axes_flat, available):
    data_dict = {'BE': expert_be[sname].dropna(), 'SC': expert_sc[sname].dropna()}
    if HAS_REAL:
        data_dict['Real'] = expert_real[sname].dropna()

    positions = list(range(len(data_dict)))
    colours = {'BE': 'steelblue', 'SC': 'darkorange', 'Real': 'grey'}

    for pos, (label, vals) in enumerate(data_dict.items()):
        if len(vals) > 0:
            vp = ax.violinplot(vals, positions=[pos], widths=0.7, showmedians=True)
            for body in vp.get('bodies', []):
                body.set_facecolor(colours[label])
                body.set_alpha(0.6)

    ax.set_xticks(positions)
    ax.set_xticklabels(list(data_dict.keys()), fontsize=8)
    ax.set_title(sname, fontsize=9)
    ax.tick_params(labelsize=7)

for j in range(len(available), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.show()

## Summary

**If both models match real data**: Both capture the essential learning dynamics.
Model selection then depends on which provides better parameter recovery / identifiability.

**If one model diverges**: Identify which stats fail — this reveals structural
differences between the models' learning mechanisms and guides model selection.

**Key comparisons**:
- Do BE and SC produce similar learning trajectories despite different mechanisms?
- Which model's expert-phase stats better match real animals?
- Do update matrices differ systematically between models?
